# 01 — Prompt Engineering Basics

Companion notebook to `01-llm-fundamentals-and-prompt-engineering.md`.

This notebook runs **entirely offline** — no API key required. We simulate an LLM with a simple
Python function that mimics next-token-style pattern completion, so you can see the *structural*
difference between zero-shot, few-shot, and chain-of-thought prompting without needing a real
model call.

> To swap in a real model later: replace `fake_llm_call()` with `AzureChatOpenAI(...).invoke(...)`
> from `langchain_openai`, using the same prompt strings built below.

## A mock LLM

This function doesn't actually understand language — it pattern-matches on the prompt to return a
canned response, purely so we can demonstrate *prompt structure* offline. It stands in for a real
call to Azure OpenAI / any chat model.

In [ ]:
import re

def fake_llm_call(prompt: str) -> str:
    """A stand-in for a real LLM call. Mimics how few-shot examples and
    chain-of-thought instructions change the *shape* of the output, without
    requiring any API key or network access.
    """
    prompt_lower = prompt.lower()

    # crude "extract the last question" so our canned answers feel responsive
    question_match = re.findall(r"question:\s*(.+)", prompt, flags=re.IGNORECASE)
    question = question_match[-1].strip() if question_match else prompt.strip()

    if "step by step" in prompt_lower or "let's think" in prompt_lower:
        return (
            "Thought: Let's break this down step by step.\n"
            "Step 1: Identify the quantities involved.\n"
            "Step 2: Determine the operation needed.\n"
            "Step 3: Compute the result.\n"
            "Final Answer: 42 (illustrative — a real model would compute this from the actual numbers)"
        )
    if "example" in prompt_lower and "answer:" in prompt_lower:
        return "Answer: Refunds are typically processed within 5-7 business days.\nSource: Refund Policy v3"
    return f"This is a generic zero-shot style answer to: '{question}'"

## 1. Zero-shot prompting

No examples, just an instruction and the question. Fast to write, and works well when the task is
common enough that the base model has seen many similar completions during training.

In [ ]:
zero_shot_prompt = """You are an internal assistant for Acme Bank.
Answer the user's question concisely.

Question: How long do refunds take?
"""

print(fake_llm_call(zero_shot_prompt))

## 2. Few-shot prompting

We show the model 1-2 example input/output pairs *in the prompt itself* before the real question.
This is the go-to technique when you need a specific output **format**, not just a correct answer —
exactly the pattern a production chatbot uses to force a consistent `Answer: ... / Source: ...`
shape the UI can reliably render.

In [ ]:
few_shot_prompt = """You are an internal assistant for Acme Bank.
Answer using this exact format, based on the example below.

Example:
Question: What are your branch hours?
Answer: Branches are open 9am-5pm, Monday to Friday.
Source: Branch Operations Handbook

Question: How long do refunds take?
"""

print(fake_llm_call(few_shot_prompt))

Notice the mock LLM returns a structured `Answer:` / `Source:` pair here — the prompt itself
*taught* it that shape, via the example. A real LLM does the same thing: it continues whatever
pattern the few-shot examples established, because next-token prediction is fundamentally about
continuing the pattern already present in the context window (see Chapter 1).

## 3. Chain-of-thought (CoT) prompting

Asking the model to reason step-by-step before giving a final answer. This matters most for
multi-step reasoning tasks — like the math word problems in the Text-to-Math agent case study
(Chapter 5) — because it gives the model more forward passes (more tokens) to work through
intermediate steps before committing to a final answer.

In [ ]:
cot_prompt = """Solve the following problem. Let's think step by step, then give a Final Answer.

Question: A train travels 60 miles in 45 minutes. What is its speed in mph?
"""

print(fake_llm_call(cot_prompt))

## 4. Optional: the real thing with `langchain_core`'s `FakeListLLM`

If `langchain_core` is installed, we can demonstrate the exact same zero-shot vs few-shot prompt
*structures* flowing through a real LangChain `PromptTemplate` + a fake (but LangChain-native) LLM —
no API key needed, and this is much closer to what you'd actually write in the chatbot codebase
(Chapter 2). If `langchain_core` isn't installed, this cell degrades gracefully.

In [ ]:
try:
    from langchain_core.language_models.fake import FakeListLLM
    from langchain_core.prompts import PromptTemplate

    # FakeListLLM just cycles through a fixed list of canned responses --
    # useful for testing chains offline, same idea as fake_llm_call() above
    # but wired into LangChain's real Runnable interface.
    fake_llm = FakeListLLM(responses=[
        "Answer: Refunds are processed within 5-7 business days.\nSource: Refund Policy v3",
    ])

    template = PromptTemplate.from_template(
        "You are an internal assistant for {client_name}.\n"
        "Question: {question}\n"
    )

    chain = template | fake_llm  # this is LCEL -- see notebook 02 for much more
    result = chain.invoke({"client_name": "Acme Bank", "question": "How long do refunds take?"})
    print(result)
except ImportError:
    print("langchain_core not installed -- install with `pip install langchain-core` to run this cell.\n"
          "The fake_llm_call() demos above already show the core concepts without any dependency.")

## Swapping in a real model

To point any of the prompts above at real Azure OpenAI instead of the mock:

```python
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_deployment="<your-deployment-name>",
    api_version="2024-05-01-preview",
    temperature=0.2,
)
# requires AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT env vars set
response = llm.invoke(few_shot_prompt)
print(response.content)
```

See `03-chatbot-architecture-azure-openai.md` for how deployments and rate limits work in Azure OpenAI.